# Explore one run

Runs `task_pipeline` once, with the harness forcing the agent through its plan's substeps in order. Shows the plan tree and the per-step log (tags, tokens, jump-ahead attempts), then runs the **independent** checker and puts the agent's local self-reports next to the global verdict.

All logic lives in `harness.py` / `checker.py`; this notebook only runs and displays. Cell 2 calls the API and costs money; run it once and re-run the later cells as often as you like.

In [1]:
import os, tempfile, time

import harness
import checker
from tasks import task_pipeline as task

for var in ("ANTHROPIC_API_KEY", "ANTHROPIC_MODEL"):
    assert os.environ.get(var), f"{var} is not set"
print("model:", os.environ["ANTHROPIC_MODEL"])

model: claude-haiku-5-5


In [2]:
# Fresh empty sandbox OUTSIDE the repo (the harness refuses anything else).
workdir = tempfile.mkdtemp(prefix=f"agentrun_{task.TASK_ID}_")

agent = harness.AnthropicAgent()
run = harness.run_task(task, agent, workdir, max_steps=60, max_actions_per_substep=8)

run_path = harness.save_run(run, f"runs/{task.TASK_ID}_{time.strftime('%Y%m%d_%H%M%S')}.json")
print("workdir: ", workdir)
print("run log: ", run_path)
print("ended by:", run["end_reason"], "after", len(run["steps"]), "steps")
print("tokens:  ", run["tokens_total"], "(planning:", run["plan_tokens"], ")")

workdir:  C:\Users\andre\AppData\Local\Temp\agentrun_pipeline_2hedro0b
run log:  runs\pipeline_20261011_131355.json
ended by: plan_complete after 41 steps
tokens:   {'input': 1171763, 'output': 23274} (planning: {'input': 1258, 'output': 1386} )


In [3]:
print(harness.format_plan(run["plan_tree"]))

goal 0: Create reader.py with the shared Order dataclass exactly as specified (field names, order, types, defaults) and the read_orders function skeleton
  0.0  Create reader.py in the working directory with the standard library imports needed (csv, datetime, dataclasses)
  0.1  Define the Order dataclass with fields order_id, customer, sku, quantity, unit_price, order_date, total=None, bulk=None in that exact order
  0.2  Write a helper that strips whitespace from header names and cell values and maps header names to column indexes regardless of column order
  0.3  Write read_orders(path) that opens the file, reads the header, and raises ValueError if any required column is missing
  0.4  Add row iteration that skips fully blank lines and keeps file order
goal 1: Implement type parsing and validation in reader.py
  1.0  Convert quantity to int and raise ValueError on parse failure
  1.1  Reject quantity values that are less than or equal to 0 with ValueError
  1.2  Convert unit_price 

In [4]:
print(harness.format_steps(run))
tags = [s["tag"] for s in run["steps"]]
jumps = sum(s["attempted_full_solution"] for s in run["steps"])
print("\nboundary steps:", tags.count("boundary"), "| internal steps:", tags.count("internal"))
print("attempted_full_solution (rejected jump-ahead writes):", jumps)

print("\nsubsteps (how each one ended):")
for sub in run["substeps"]:
    print(f"  {sub['goal_index']}.{sub['substep_index']}  actions={sub['actions']:<2} "
          f"ended_by={sub['ended_by']}")

step   g.s tag       tool       jump  tok_in tok_out local_ok*  intent
   0   0.0 internal  write_file         3446     168       yes  Create reader.py with the standard library imports
   1   0.1 internal  write_file         3882     253       yes  Define the Order dataclass with exact fields, orde
   2   0.2 internal  write_file JUMP    4382     419        NO  Add helpers that strip cell values and map header 
   3   0.2 internal  write_file         4894     441       yes  Add a helper that maps stripped header names to co
   4   0.3 internal  write_file         5605     577       yes  Write read_orders skeleton that opens the file, re
   5   0.4 internal  write_file         6429    1028       yes  Add row iteration that skips blank lines and keeps
   6   1.0 boundary  write_file         7709     856       yes  Convert quantity to int in read_orders and raise V
   7   1.1 internal  write_file         8785     820       yes  Reject quantity values less than or equal to 0 wit
   8   1.

In [5]:
# Independent global check: hidden pytest suite, exit code only. Sees only the workdir.
global_ok = checker.global_success(task, workdir, show_output=True)

.........                                                                [100%]
9 passed in 4.96s



In [6]:
reports = [s["local_check_self_report_untrusted"] for s in run["steps"]]
# "All passed" needs every step reported AND reported True; a missing report is not a pass.
local_all_ok = bool(reports) and all(r is True for r in reports)

print(f"local steps all passed? (agent self-report, UNTRUSTED): {local_all_ok}"
      f"   [yes={reports.count(True)} no={reports.count(False)} unreported={reports.count(None)}]")
print(f"global success?        (hidden tests, independent):    {global_ok}")
print(f"jump-ahead attempts:   {jumps}")
if local_all_ok and not global_ok:
    print("\n>>> LOCAL-PASS / GLOBAL-FAIL: the case this project hunts for.")

local steps all passed? (agent self-report, UNTRUSTED): False   [yes=36 no=5 unreported=0]
global success?        (hidden tests, independent):    True
jump-ahead attempts:   3
